# Day 3 — Dataset Preparation, Preprocessing, and Baseline Model

**Objective:** Build a reproducible supervised-learning workflow for predicting customer churn.

This notebook loads the source CSV independently, applies the documented `TotalCharges` cleaning assumption, separates features and target, creates training/validation/test partitions, and builds a Logistic Regression baseline.

**Evaluation note:** An earlier version of this notebook used the original test partition for an exploratory threshold experiment. Because that partition was used during model development, it must not be described as an untouched final test set. Day 4 uses validation data for comparisons; no final test score is claimed here.

## 1. Imports and Load Data

This notebook must run independently; it does not depend on variables created in `01_data_inspection.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

df = pd.read_csv("../data/telco-customer-churn.csv")
print("Dataset shape before cleaning:", df.shape)

## 2. Apply the Documented Cleaning Step

In the inspection notebook, 11 `TotalCharges` entries were found to contain whitespace rather than ordinary null values. These were associated with zero-tenure customers. We provisionally replace blank values with zero, as documented in the EDA notebook; validate this assumption against the dataset definition before production use.

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].str.strip().replace("", pd.NA),
    errors="raise"
)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Missing values after cleaning:", df.isnull().sum().sum())
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Dataset shape after cleaning:", df.shape)

## 3. Define Features (`X`) and Target (`y`)

`Churn` is the outcome we want to predict, so it must not appear among the input features. `customerID` is an identifier and is excluded from the initial model inputs.

In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution (%):")
print(y.value_counts(normalize=True).mul(100).round(2))

## 4. Create Training, Validation, and Test Sets

We first reserve 20% of the records as a test partition, then split the remaining 80% into training and validation subsets. The resulting proportions are approximately 60% training, 20% validation, and 20% test.

**Important limitation:** This notebook originally explored a threshold using a test split created with `random_state=42`. The split below retains that seed for reproducibility, so the test partition is not new or untouched. Do not use it to choose models or thresholds, and do not present it as an independent final evaluation.

### Split Procedure

- **Training set (60%):** used to fit the model and preprocessing steps.
- **Validation set (20%):** used to compare model configurations and select a threshold.
- **Test partition (20%):** retained for reference only in this notebook; it is not treated as an untouched final test set because it was used in an earlier exploratory experiment.

Stratified splitting preserves the churn/non-churn proportions approximately in each subset.

In [ ]:
# Reserve 20% as the test partition; note that this partition was used in an earlier exploratory experiment.
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split the remaining 80% into training and validation sets.
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

print("\nChurn proportion in each set:")
print("Training:", y_train.value_counts(normalize=True))
print("Validation:", y_val.value_counts(normalize=True))
print("Test:", y_test.value_counts(normalize=True))

## 5. Identify Feature Groups

We use data types to identify the initial groups, then explicitly treat `SeniorCitizen` as a categorical indicator. It is stored as 0/1, but those values represent categories rather than a continuous measurement. Treating it as numeric could also be defensible; this notebook makes the chosen treatment explicit.

In [ ]:
numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numerical_features.remove("SeniorCitizen")
categorical_features.append("SeniorCitizen")

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

## 6. Build the Preprocessor

`StandardScaler` learns each numerical feature's mean and standard deviation from training data. `OneHotEncoder` converts categorical values into indicator columns. `handle_unknown="ignore"` prevents an unseen category from causing an error during transformation.

We will fit this preprocessor through the pipeline, using training data only.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## 7. Build and Train the Baseline Pipeline

The pipeline combines preprocessing and Logistic Regression. Passing the original `X_train` to `.fit()` lets the pipeline fit the transformations on training data and then train the classifier on the transformed features.

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

## Day 4 — Model Evaluation and Error Analysis

### 1. Generate Validation Predictions

Use the fitted pipeline to predict churn labels for the validation set. These predictions will help us evaluate the model and compare classification thresholds without repeatedly consulting the test set.

In [ ]:
y_val_pred = model.predict(X_val)

In [ ]:
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, pos_label="Yes"))
print("Recall:", recall_score(y_val, y_val_pred, pos_label="Yes"))
print("F1-score:", f1_score(y_val, y_val_pred, pos_label="Yes"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred, labels=["No", "Yes"]))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

### 2.1 Validation Baseline — Interpretation

The baseline Logistic Regression model was evaluated on the validation set using the default classification threshold of 0.50.

- **Accuracy:** 80.27%
- **Precision (churn):** 65.79%
- **Recall (churn):** 53.48%
- **F1-score (churn):** 59.00%

The confusion matrix contains 931 true negatives, 104 false positives, 174 false negatives, and 200 true positives.

The model correctly identifies many customers who stay, but it misses 174 of the 374 customers who actually churned in the validation set. Improving churn detection while controlling false positives is therefore a key objective for subsequent experiments.

These validation results establish the baseline for comparing classification thresholds and model configurations.

### 2.2 Generate Validation Probabilities

Logistic Regression can estimate a probability for each class. We use the predicted probability of churn to evaluate alternative classification thresholds on the validation set.

In [ ]:
y_val_prob = model.predict_proba(X_val)[:, 1]

### 2.3 Compare Classification Thresholds

Logistic Regression produces predicted probabilities, which are converted into class labels using a decision threshold.

A customer is predicted to churn when their predicted churn probability is greater than or equal to the threshold.

Lower thresholds generally increase recall while reducing precision. We will compare several thresholds using validation data and select an operating point based on the business objective.

In [ ]:
thresholds = [0.30, 0.40, 0.50]

threshold_results = []

for threshold in thresholds:
    y_val_pred_threshold = np.where(
        y_val_prob >= threshold,
        "Yes",
        "No"
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, y_val_pred_threshold, pos_label="Yes"
        ),
        "Recall": recall_score(
            y_val, y_val_pred_threshold, pos_label="Yes"
        ),
        "F1-score": f1_score(
            y_val, y_val_pred_threshold, pos_label="Yes"
        )
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df

### 2.4 Threshold Comparison — Findings

Three classification thresholds were evaluated using validation probabilities.

- **Threshold 0.30:** Precision = 53.83%, Recall = 75.13%, F1-score = 62.72%.
- **Threshold 0.40:** Precision = 59.51%, Recall = 64.44%, F1-score = 61.87%.
- **Threshold 0.50:** Precision = 65.79%, Recall = 53.48%, F1-score = 59.00%.

Lowering the threshold increased recall while reducing precision. Among the tested thresholds, 0.30 achieved the highest recall and F1-score, while 0.50 achieved the highest precision.

Threshold 0.30 is a candidate for further investigation, not yet a final deployment decision. The appropriate threshold depends on retention-campaign costs, campaign effectiveness, and the relative costs of false positives and false negatives.

### 2.5 Confusion Matrix Comparison

A confusion matrix shows the counts of true negatives, false positives, false negatives, and true positives for a classification threshold.

Comparing confusion matrices across thresholds helps quantify the operational trade-off between identifying more customers who churn and flagging customers who would stay.

In [ ]:
for threshold in [0.30, 0.40, 0.50]:
    y_val_pred_threshold = np.where(
        y_val_prob >= threshold,
        "Yes",
        "No"
    )

    cm = confusion_matrix(
        y_val,
        y_val_pred_threshold,
        labels=["No", "Yes"]
    )

    print(f"\nThreshold: {threshold:.2f}")
    print(cm)

### 2.6 Confusion Matrix Comparison — Findings

The confusion matrices show how the classification threshold affects the number of customers flagged for potential churn.

- At threshold 0.30, the model identifies 281 churners, misses 93, and produces 241 false positives.
- At threshold 0.40, the model identifies 241 churners, misses 133, and produces 164 false positives.
- At threshold 0.50, the model identifies 200 churners, misses 174, and produces 104 false positives.

Lowering the threshold from 0.50 to 0.30 identifies 81 additional churners and reduces missed churners by 81, while increasing false positives by 137. The number of customers flagged rises from 304 to 522.

The best operating threshold depends on retention-campaign costs and benefits. These validation results support further investigation but do not establish campaign profitability.

### 2.7 Ranking Metrics: ROC-AUC and Average Precision

ROC-AUC measures the model's ability to rank customers who churn above customers who stay across different classification thresholds.

Average Precision summarizes the precision-recall relationship across thresholds. It is particularly useful when the positive class is less common than the negative class.

Both metrics use predicted probabilities rather than class labels. They evaluate ranking performance and do not, by themselves, determine the best deployment threshold.

In [ ]:
roc_auc = roc_auc_score(
    (y_val == "Yes").astype(int),
    y_val_prob
)

average_precision = average_precision_score(
    (y_val == "Yes").astype(int),
    y_val_prob
)

print(f"Validation ROC-AUC: {roc_auc:.4f}")
print(f"Validation Average Precision: {average_precision:.4f}")

In [ ]:
RocCurveDisplay.from_predictions(
    (y_val == "Yes").astype(int),
    y_val_prob
)
plt.title("ROC Curve — Validation Set")
plt.show()

PrecisionRecallDisplay.from_predictions(
    (y_val == "Yes").astype(int),
    y_val_prob
)
plt.title("Precision-Recall Curve — Validation Set")
plt.show()

### 2.8 Ranking Metrics — Findings

The baseline Logistic Regression model achieved the following validation ranking metrics:

- **ROC-AUC:** 0.8360
- **Average Precision:** 0.6424

The ROC-AUC suggests that the model can distinguish churners from non-churners reasonably well by ranking their predicted probabilities. Average Precision is substantially above the approximate positive-class prevalence of 26.5%, indicating useful precision-recall ranking performance.

These metrics evaluate ranking across thresholds. They do not identify the best deployment threshold, establish probability calibration, or prove that a retention campaign will be profitable.

The next step is to inspect the ROC and Precision-Recall curves and investigate the customers the model misclassifies.

## 2. Error Analysis

Error analysis examines individual predictions to identify patterns in false positives and false negatives.

We will compare the actual churn labels with the model's predicted labels and probabilities on the validation set. This analysis can reveal which types of customers the model struggles to classify and help guide further experiments.

In [ ]:
error_analysis = X_val.copy()

error_analysis["Actual_Churn"] = y_val
error_analysis["Predicted_Churn"] = y_val_pred
error_analysis["Churn_Probability"] = y_val_prob

error_analysis["Error_Type"] = np.select(
    [
        (error_analysis["Actual_Churn"] == "Yes") &
        (error_analysis["Predicted_Churn"] == "No"),

        (error_analysis["Actual_Churn"] == "No") &
        (error_analysis["Predicted_Churn"] == "Yes")
    ],
    [
        "False Negative",
        "False Positive"
    ],
    default="Correct"
)

error_analysis.head()

In [ ]:
error_analysis["Error_Type"].value_counts()

In [ ]:
false_negatives = error_analysis[
    error_analysis["Error_Type"] == "False Negative"
]

false_negatives.head()

In [ ]:
false_positives = error_analysis[
    error_analysis["Error_Type"] == "False Positive"
]

false_positives.head()

### 2.1 Investigating False Negatives by Contract

Contract type was strongly associated with churn during exploratory data analysis. We will examine the contract types of customers the model failed to identify, then compare these patterns with the full validation set.

This analysis is exploratory: it helps identify potential weaknesses but does not establish why an individual prediction was wrong.

In [ ]:
false_negatives["Contract"].value_counts()

In [ ]:
contract_errors = error_analysis.groupby("Contract").apply(
    lambda group: (
        (group["Error_Type"] == "False Negative").sum()
        / (group["Actual_Churn"] == "Yes").sum()
    ),
    include_groups=False
)

contract_errors.sort_values(ascending=False)

In [ ]:
contract_error_summary = pd.DataFrame({
    "Actual_Churners": error_analysis[
        error_analysis["Actual_Churn"] == "Yes"
    ].groupby("Contract").size(),

    "False_Negatives": error_analysis[
        error_analysis["Error_Type"] == "False Negative"
    ].groupby("Contract").size(),

    "False_Negative_Rate": contract_errors
})

contract_error_summary

In [ ]:
contract_churn_rate = (
    error_analysis.groupby("Contract")["Actual_Churn"]
    .apply(lambda values: (values == "Yes").mean())
)

contract_error_summary["Churn_Rate"] = contract_churn_rate

contract_error_summary

### 2.2 Inspect Predicted Probabilities of Missed Churners

The model missed all 37 actual churners with one-year contracts and all 13 actual churners with two-year contracts in the validation sample at the default threshold of 0.50.

Because these groups contain relatively few churners, their false-negative rates should be interpreted cautiously. We will inspect the predicted churn probabilities of these customers to determine whether the missed cases are close to the classification threshold or receive very low risk scores.

In [ ]:
false_negatives.groupby("Contract")["Churn_Probability"].agg(
    ["count", "min", "mean", "median", "max"]
)

### 2.3 Compare Correctly Identified and Missed Churners

We will compare the characteristics of actual churners who were correctly identified by the model with those who were classified as non-churners.

The purpose is to identify potential patterns in prediction errors, including differences in tenure, monthly charges, total charges, and contract type. These comparisons are exploratory and do not establish causation.

In [ ]:
actual_churners = error_analysis[
    error_analysis["Actual_Churn"] == "Yes"
].copy()

actual_churners["Detection_Status"] = np.where(
    actual_churners["Predicted_Churn"] == "Yes",
    "Correctly Identified",
    "Missed"
)

actual_churners.groupby("Detection_Status")[
    ["tenure", "MonthlyCharges", "TotalCharges"]
].agg(["count", "mean", "median"])

### 2.4 Error Analysis — Numerical Feature Findings

Comparing correctly identified churners with missed churners revealed the following patterns:

- Correctly identified churners had a mean tenure of 12.50 months and a median tenure of 8 months.
- Missed churners had a mean tenure of 28.57 months and a median tenure of 23.5 months.
- Mean monthly charges were approximately 82.98 for correctly identified churners and 67.58 for missed churners.
- Mean total charges were approximately 1,138.42 for correctly identified churners and 2,356.24 for missed churners.

These descriptive results suggest that the baseline model identifies short-tenure, higher-monthly-charge churners more effectively than some longer-tenure, lower-monthly-charge churners. Further investigation is needed before attributing these errors to specific features or making modeling changes.

### 2.5 Error Analysis — Contract Patterns

We will compare contract types among actual churners who were correctly identified and those who were missed by the baseline model.

This analysis builds on the earlier finding that the model missed all observed validation churners with one-year and two-year contracts at the default threshold. We will compare the groups' distributions while keeping sample sizes in mind.

In [ ]:
pd.crosstab(
    actual_churners["Contract"],
    actual_churners["Detection_Status"],
    normalize="index"
).round(3).mul(100)

### 2.6 Contract-Level Error Analysis — Findings

Among actual churners in the validation set, the baseline model correctly identified 61.7% of month-to-month customers and missed 38.3%.

The model missed all observed churners in the one-year and two-year contract groups at the default threshold of 0.50.

The longer-contract groups contain fewer actual churners, so their observed detection rates should be interpreted cautiously. These results suggest that the baseline model may not generalize equally well across contract groups and motivate further controlled experiments.

## 3. Controlled Model Improvement

### 4.1 Experiment: Class-Weighted Logistic Regression

The baseline model may favor the majority class because non-churners outnumber churners. We will test Logistic Regression with `class_weight="balanced"` to increase the relative penalty for misclassifying the minority class.

Only the classifier setting will change. We will keep the training and validation splits, preprocessing approach, and evaluation metrics consistent so that the comparison isolates the effect of class weighting.

In [ ]:
from sklearn.base import clone

In [ ]:
weighted_model = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
weighted_model.fit(X_train, y_train)

### 3.1 Evaluate the Class-Weighted Model

Evaluate the class-weighted Logistic Regression model on the same validation set used for the baseline. Compare precision, recall, F1-score, and confusion matrix to determine how class weighting affects churn detection.

In [ ]:
y_val_pred_weighted = weighted_model.predict(X_val)

In [ ]:
print("Weighted Model — Validation Results")
print("Accuracy:", accuracy_score(y_val, y_val_pred_weighted))
print(
    "Precision:",
    precision_score(y_val, y_val_pred_weighted, pos_label="Yes")
)
print(
    "Recall:",
    recall_score(y_val, y_val_pred_weighted, pos_label="Yes")
)
print(
    "F1-score:",
    f1_score(y_val, y_val_pred_weighted, pos_label="Yes")
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred_weighted,
        labels=["No", "Yes"]
    )
)

In [ ]:
model_comparison = pd.DataFrame([
    {
        "Model": "Baseline Logistic Regression",
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(
            y_val, y_val_pred, pos_label="Yes"
        ),
        "Recall": recall_score(
            y_val, y_val_pred, pos_label="Yes"
        ),
        "F1-score": f1_score(
            y_val, y_val_pred, pos_label="Yes"
        )
    },
    {
        "Model": "Class-Weighted Logistic Regression",
        "Accuracy": accuracy_score(y_val, y_val_pred_weighted),
        "Precision": precision_score(
            y_val, y_val_pred_weighted, pos_label="Yes"
        ),
        "Recall": recall_score(
            y_val, y_val_pred_weighted, pos_label="Yes"
        ),
        "F1-score": f1_score(
            y_val, y_val_pred_weighted, pos_label="Yes"
        )
    }
])

model_comparison.round(4)

### 3.2 Class-Weighted Logistic Regression — Findings

The class-weighted Logistic Regression model was evaluated on the same validation set as the baseline, using the default threshold of 0.50.

| Metric | Baseline | Class-weighted |
|---|---:|---:|
| Accuracy | 80.27% | 74.80% |
| Precision | 65.79% | 51.66% |
| Recall | 53.48% | 78.88% |
| F1-score | 59.00% | 62.43% |

Class weighting increased recall by 25.40 percentage points and F1-score by approximately 3.43 percentage points, while reducing accuracy and precision.

The weighted model is a promising candidate for a retention strategy that prioritizes identifying more churners. However, the additional false positives must be considered before selecting a model for deployment. This comparison is based on validation performance and is not a final test evaluation.

### 3.3 Threshold Analysis — Class-Weighted Model

The class-weighted model changes how Logistic Regression treats errors on each class. Its predicted probabilities may therefore differ from those of the baseline model.

We will evaluate multiple thresholds on validation data to compare precision, recall, and F1-score. The test set will not be used to select a threshold.

In [ ]:
y_val_prob_weighted = weighted_model.predict_proba(X_val)[:, 1]

In [ ]:
thresholds = [0.30, 0.40, 0.50]
weighted_threshold_results = []

for threshold in thresholds:
    y_val_pred_weighted_threshold = np.where(
        y_val_prob_weighted >= threshold,
        "Yes",
        "No"
    )

    weighted_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, y_val_pred_weighted_threshold, pos_label="Yes"
        ),
        "Recall": recall_score(
            y_val, y_val_pred_weighted_threshold, pos_label="Yes"
        ),
        "F1-score": f1_score(
            y_val, y_val_pred_weighted_threshold, pos_label="Yes"
        )
    })

weighted_threshold_results_df = pd.DataFrame(
    weighted_threshold_results
)

weighted_threshold_results_df

### 3.4 Class-Weighted Threshold Analysis — Findings

The class-weighted model was evaluated at thresholds 0.30, 0.40, and 0.50 using validation probabilities.

- **Threshold 0.30:** Precision = 41.99%, Recall = 91.18%, F1-score = 57.50%.
- **Threshold 0.40:** Precision = 45.89%, Recall = 85.03%, F1-score = 59.07%.
- **Threshold 0.50:** Precision = 51.66%, Recall = 78.88%, F1-score = 62.43%.

Lowering the threshold increased recall but reduced precision. Among the tested weighted-model thresholds, 0.50 achieved the highest F1-score, while 0.30 achieved the highest recall.

The weighted model offers a stronger churn-detection trade-off than the baseline at threshold 0.50, but its additional false positives may increase retention-campaign costs. The final choice should be based on the business objective and validation evidence, with final performance evaluated on data not used for model selection.

### 3.5 Ranking Metric Comparison

Compare the baseline and class-weighted Logistic Regression models using ROC-AUC and Average Precision on the validation set.

These metrics evaluate how well each model ranks customers by churn risk across different thresholds. They complement precision, recall, and F1-score, which depend on a specific classification threshold.

In [ ]:
y_val_binary = (y_val == "Yes").astype(int)

ranking_comparison = pd.DataFrame([
    {
        "Model": "Baseline Logistic Regression",
        "ROC-AUC": roc_auc_score(y_val_binary, y_val_prob),
        "Average Precision": average_precision_score(
            y_val_binary, y_val_prob
        )
    },
    {
        "Model": "Class-Weighted Logistic Regression",
        "ROC-AUC": roc_auc_score(
            y_val_binary, y_val_prob_weighted
        ),
        "Average Precision": average_precision_score(
            y_val_binary, y_val_prob_weighted
        )
    }
])

ranking_comparison.round(4)

### 3.6 Ranking Metric Comparison — Findings

The baseline and class-weighted Logistic Regression models achieved nearly identical validation ranking metrics.

| Metric | Baseline | Class-weighted |
|---|---:|---:|
| ROC-AUC | 0.8360 | 0.8363 |
| Average Precision | 0.6424 | 0.6425 |

The differences were very small. Although class weighting substantially increased recall at threshold 0.50, it did not produce a meaningful observed improvement in ROC-AUC or Average Precision in this experiment.

This suggests that the main observed benefit of class weighting was a change in classification behavior rather than a substantial improvement in ranking performance. Further evaluation is required before selecting the final model.

## 4. Day 4 Summary and Next Steps

### 4.1 What We Accomplished

- Created training, validation, and test partitions.
- Fit a Logistic Regression pipeline using training data only.
- Evaluated validation performance using accuracy, precision, recall, F1-score, confusion matrices, ROC-AUC, and Average Precision.
- Compared thresholds of 0.30, 0.40, and 0.50 on validation data.
- Investigated false positives, false negatives, contract-level patterns, tenure, monthly charges, and total charges.
- Compared baseline Logistic Regression with a class-weighted version.

### 4.2 Key Results From This Run

The baseline validation ROC-AUC was approximately 0.8360 and Average Precision was approximately 0.6424. At threshold 0.50, the baseline achieved about 80.27% accuracy, 65.79% precision, 53.48% recall, and 59.00% F1-score.

At threshold 0.50, the class-weighted model achieved about 74.80% accuracy, 51.66% precision, 78.88% recall, and 62.43% F1-score. Its ROC-AUC and Average Precision were nearly unchanged from the baseline.

**Re-run all cells before relying on these figures.** They are outputs from the previous execution and will be cleared in this reviewed notebook.

### 4.3 Interpretation

Class weighting substantially increased recall while reducing precision and accuracy. In this experiment, it changed threshold-based classification behavior much more than it changed ranking metrics. Whether that trade-off is desirable depends on the costs and benefits of contacting customers.

The error analysis also showed that the baseline missed all observed churners in the one-year and two-year contract groups in the validation sample. These groups were small, so the rates should be interpreted cautiously. Missed churners had longer tenure and lower mean monthly charges than correctly identified churners; these are associations, not proven causes.

### 4.4 Limitations

- Only a small set of thresholds was tested.
- Business costs for false positives and false negatives have not been quantified.
- The original test partition was used for an exploratory threshold experiment. It is therefore **not an untouched final test set**, and no final test-performance claim is made in this notebook.
- The notebook's current split is reproducible, but a rigorous final assessment should use a genuinely untouched dataset or a carefully planned evaluation procedure that accounts for the earlier test-set exploration.

### 4.5 Next Steps

1. Define the business objective and acceptable precision-recall trade-off.
2. Continue model selection using training and validation data only.
3. Consider cross-validation for more stable model comparisons.
4. Establish a defensible final evaluation strategy before reporting final test performance.
5. Re-run the notebook from top to bottom and confirm all cells execute in order.

## Historical Test-Set Experiment — Not Re-run

An earlier version of this notebook evaluated predictions and a 0.30 threshold on the test partition. Those cells have been removed from the active workflow to prevent accidental use of test results for further model selection. The historical experiment is acknowledged above as a limitation; it is not a final performance estimate.

# Day 5 — Systematic Model Improvement

## 1. Experiment Objective

The objective is to investigate whether feature engineering improves the predictive performance of the baseline Logistic Regression model.

We will change one component at a time, fit each candidate model using training data only, and compare performance using the same validation set.

### Baseline Reference

The baseline model achieved a validation ROC-AUC of approximately 0.8360 and Average Precision of 0.6424. Class weighting substantially increased recall at threshold 0.50 but produced almost no change in ranking metrics.

### Evaluation Principles

- Keep the training and validation partitions consistent across experiments.
- Fit preprocessing and feature transformations using training data only.
- Use validation metrics to compare candidate models.
- Avoid using the test partition to select features, models, or thresholds.
- Record each experiment and its results before deciding whether to retain the change.

In [ ]:
from sklearn.preprocessing import SplineTransformer

In [ ]:
spline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "tenure_spline",
            Pipeline(steps=[
                ("spline", SplineTransformer(n_knots=4, degree=3, include_bias=False)),
                ("scale", StandardScaler())]), ["tenure"]),
        ("num", StandardScaler(),["MonthlyCharges", "TotalCharges"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
spline_model = Pipeline(
    steps=[
        ("preprocessor", spline_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [ ]:
spline_model.fit(X_train, y_train)

In [ ]:
spline_val_prob = spline_model.predict_proba(X_val)[:, 1]

In [ ]:
spline_roc_auc = roc_auc_score(y_val, spline_val_prob)
spline_avg_precision = average_precision_score(
    (y_val == "Yes").astype(int),
    spline_val_prob
)

print(f"Spline Model ROC-AUC: {spline_roc_auc:.4f}")
print(f"Spline Model Average Precision: {spline_avg_precision:.4f}")

print(f"Baseline ROC-AUC: {roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]):.4f}")
print(f"Baseline Average Precision: {average_precision_score((y_val == 'Yes').astype(int), model.predict_proba(X_val)[:, 1]):.4f}")

### Experiment 1 — Nonlinear Tenure Feature Engineering

**Hypothesis:** The relationship between customer tenure and churn may be nonlinear. A spline transformation may improve Logistic Regression's predictive ranking.

**Change:** Applied `SplineTransformer(n_knots=4, degree=3, include_bias=False)` to `tenure`. The other numerical features were standardized, and categorical features were one-hot encoded as in the baseline.

**Validation results:**

| Metric | Baseline | Spline Model | Change |
|---|---:|---:|---:|
| ROC-AUC | 0.8360 | 0.8407 | +0.0047 |
| Average Precision | 0.6424 | 0.6539 | +0.0115 |

**Interpretation:** Both validation ranking metrics improved. The result supports further investigation of nonlinear tenure effects, but the improvement is modest and requires additional evaluation.

**Decision:** Retain the spline model as a candidate for comparison. Do not replace the baseline permanently based on this experiment alone.

**Limitations:** Only one spline configuration was tested. The result comes from a single validation partition and does not establish statistical significance or guarantee performance on new data.

In [ ]:
# Generate predictions from both models
baseline_val_pred = model.predict(X_val)
spline_val_pred = spline_model.predict(X_val)

# Calculate metrics for the baseline
baseline_precision = precision_score(
    y_val, baseline_val_pred, pos_label="Yes"
)
baseline_recall = recall_score(
    y_val, baseline_val_pred, pos_label="Yes"
)
baseline_f1 = f1_score(
    y_val, baseline_val_pred, pos_label="Yes"
)

# Calculate metrics for the spline model
spline_precision = precision_score(
    y_val, spline_val_pred, pos_label="Yes"
)
spline_recall = recall_score(
    y_val, spline_val_pred, pos_label="Yes"
)
spline_f1 = f1_score(
    y_val, spline_val_pred, pos_label="Yes"
)

# Display the comparison
print("Baseline Model")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1-score:  {baseline_f1:.4f}")

print("\nSpline Model")
print(f"Precision: {spline_precision:.4f}")
print(f"Recall:    {spline_recall:.4f}")
print(f"F1-score:  {spline_f1:.4f}")

In [ ]:
baseline_cm = confusion_matrix(
    y_val, baseline_val_pred, labels=["No", "Yes"]
)

spline_cm = confusion_matrix(
    y_val, spline_val_pred, labels=["No", "Yes"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(
    confusion_matrix=baseline_cm,
    display_labels=["No", "Yes"]
).plot(ax=axes[0], values_format="d", colorbar=False)
axes[0].set_title("Baseline Model")

ConfusionMatrixDisplay(
    confusion_matrix=spline_cm,
    display_labels=["No", "Yes"]
).plot(ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("Spline Model")

plt.tight_layout()
plt.show()

**Confusion Matrix Comparison at Threshold 0.50**

| Outcome | Baseline | Spline Model |
|---|---:|---:|
| True negatives | 931 | 940 |
| False positives | 104 | 95 |
| False negatives | 174 | 175 |
| True positives | 200 | 199 |

**Interpretation:** The spline model reduced false positives by 9 but increased false negatives by 1. It therefore produced slightly fewer false alarms without improving churn detection at this threshold.

**Decision:** Retain the spline model as a candidate because its validation ROC-AUC, Average Precision, precision, and F1-score improved. Further comparisons are needed before selecting a final model.

In [ ]:
def evaluate_thresholds(y_true, probabilities, thresholds):
    results = []

    for threshold in thresholds:
        predictions = [
            "Yes" if probability >= threshold else "No"
            for probability in probabilities
        ]

        results.append({
            "Threshold": threshold,
            "Precision": precision_score(
                y_true, predictions, pos_label="Yes"
            ),
            "Recall": recall_score(
                y_true, predictions, pos_label="Yes"
            ),
            "F1": f1_score(
                y_true, predictions, pos_label="Yes"
            ),
            "Flagged Customers": sum(
                prediction == "Yes" for prediction in predictions
            )
        })

    return pd.DataFrame(results)

In [ ]:
thresholds = [0.30, 0.40, 0.50]

baseline_threshold_results = evaluate_thresholds(
    y_val, model.predict_proba(X_val)[:, 1], thresholds
)

spline_threshold_results = evaluate_thresholds(
    y_val, spline_val_prob, thresholds
)

print("Baseline Model")
display(baseline_threshold_results.round(4))

print("Spline Model")
display(spline_threshold_results.round(4))

**Threshold Analysis**

The baseline and spline models were compared at thresholds of 0.30, 0.40, and 0.50 using the same validation partition.

At threshold 0.40, the spline model achieved precision of 0.6129, recall of 0.6604, and F1-score of 0.6358, compared with 0.5951, 0.6444, and 0.6187 for the baseline.

The spline model therefore improved both precision and recall at this threshold, while flagging 403 customers compared with 405 for the baseline.

**Interpretation:** Threshold 0.40 provides the strongest F1-score among the three tested thresholds for the spline model. The improvement is promising, but the threshold is not yet selected as the final operational threshold.

**Decision:** Retain the spline model as the leading candidate for further validation. Threshold selection should ultimately reflect the relative costs of false positives and false negatives.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import average_precision_score
import numpy as np

# Use the same three stratified folds for both models.
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Custom scorer: scikit-learn passes the fitted estimator, X, and y.
# Explicitly score "Yes" as the positive churn class.
def average_precision_yes(estimator, X, y_true):
    probabilities = estimator.predict_proba(X)[:, 1]
    y_true_binary = (np.asarray(y_true) == "Yes").astype(int)
    return average_precision_score(y_true_binary, probabilities)

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": average_precision_yes
}

baseline_cv = cross_validate(
    model, X_dev, y_dev,
    cv=cv, scoring=scoring,
    n_jobs=1, error_score="raise"
)

spline_cv = cross_validate(
    spline_model, X_dev, y_dev,
    cv=cv, scoring=scoring,
    n_jobs=1, error_score="raise"
)

print("Baseline ROC-AUC:", baseline_cv["test_roc_auc"])
print("Baseline Average Precision:", baseline_cv["test_average_precision"])
print("Spline ROC-AUC:", spline_cv["test_roc_auc"])
print("Spline Average Precision:", spline_cv["test_average_precision"])


In [ ]:
cv_results = pd.DataFrame({
    "Model": ["Baseline", "Spline"],
    "Mean ROC-AUC": [
        baseline_cv["test_roc_auc"].mean(),
        spline_cv["test_roc_auc"].mean()
    ],
    "ROC-AUC Std": [
        baseline_cv["test_roc_auc"].std(),
        spline_cv["test_roc_auc"].std()
    ],
    "Mean Average Precision": [
        baseline_cv["test_average_precision"].mean(),
        spline_cv["test_average_precision"].mean()
    ],
    "AP Std": [
        baseline_cv["test_average_precision"].std(),
        spline_cv["test_average_precision"].std()
    ]
})

display(cv_results.round(4))